# BirdCLEF+ 2026 — Inference (LB 0.842 pipeline)

Loads:
- **Perch v2 ONNX** from `data/perch/perch_v2_no_dft.onnx`
- **10 SED heads** from `checkpoints/`:
  - 5 Phase 4 seeds: `model_v8_seed{42..46}.pt`
  - 5 Phase 6 folds: `model_v9_fold{0..4}.pt`

Pipeline per 60-second soundscape:

```
60s audio
   ↓ slice into 12 × 5s windows
   ↓ Perch ONNX (batched over 12) → spatial_embedding (12, 16, 4, 1536)
   ↓ mean-pool freq                → (12, 16, 1536)
   ↓ 10 SED heads, average logits → (12, 234) clip logits
   ↓ Gaussian smooth across 12 windows (in logit space)
   ↓ sigmoid → (12, 234) probabilities
   ↓ rows: {filename}_5, {filename}_10, ..., {filename}_60
```

Locally, `test_soundscapes/` is empty (Kaggle reveals real test audio only during scored submission),
so we fall back to 5 files from `train_soundscapes/` to verify the pipeline.


## 1. Setup


In [ ]:
import os, time
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
import torch.nn as nn
import onnxruntime as ort
from scipy.ndimage import convolve1d
from tqdm.auto import tqdm

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Device:", DEVICE)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data" / "birdclef-2026"
PERCH_PATH   = PROJECT_ROOT / "data" / "perch" / "perch_v2_no_dft.onnx"
CKPT_DIR     = PROJECT_ROOT / "checkpoints"
OUT_PATH     = PROJECT_ROOT / "submission.csv"

assert PERCH_PATH.exists(), f"Missing {PERCH_PATH}"


## 2. Load Perch ONNX

Take the `spatial_embedding` output, not the global `embedding`. The SED head
expects per-timestep features.


In [ ]:
sess = ort.InferenceSession(str(PERCH_PATH), providers=["CPUExecutionProvider"])
INPUT_NAME  = sess.get_inputs()[0].name
SPATIAL_IDX = next(i for i, o in enumerate(sess.get_outputs()) if o.name == "spatial_embedding")
print(f"Perch loaded. spatial_embedding output idx={SPATIAL_IDX}")


## 3. SED head class (same as training)

We need this defined to load checkpoint weights.


In [ ]:
class PerchSEDHead(nn.Module):
    def __init__(self, embed_dim=1536, hidden_dim=512, num_classes=234, dropout=0.3):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)
        self.bottleneck = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = self.norm(x)
        x = self.bottleneck(x)
        x = x.transpose(1, 2)
        att = torch.softmax(torch.tanh(self.att(x)), dim=2)
        cla = self.cla(x)
        return (att * cla).sum(dim=2)


## 4. Load all 10 SED heads (5 v8 seeds + 5 v9 folds)

Each head's checkpoint carries the `species` list and `label_to_idx` — they're all
identical, so we just grab them from the first.


In [ ]:
SEED_CKPTS = sorted(CKPT_DIR.glob("model_v8_seed*.pt"))
FOLD_CKPTS = sorted(CKPT_DIR.glob("model_v9_fold*.pt"))
ALL_CKPTS  = SEED_CKPTS + FOLD_CKPTS

assert len(ALL_CKPTS) > 0, f"No checkpoints found in {CKPT_DIR}. Run 01_train.ipynb first."
print(f"Found {len(SEED_CKPTS)} Phase 4 seeds + {len(FOLD_CKPTS)} Phase 6 folds = {len(ALL_CKPTS)} models")

heads = []
species, label_to_idx, NUM_CLASSES, EMBED_DIM = None, None, None, None
for ckpt_path in ALL_CKPTS:
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    if species is None:
        species       = ckpt["species"]
        label_to_idx  = ckpt["label_to_idx"]
        NUM_CLASSES   = ckpt["num_classes"]
        EMBED_DIM     = ckpt["embed_dim"]
    h = PerchSEDHead(EMBED_DIM, **ckpt["head_config"], num_classes=NUM_CLASSES).to(DEVICE)
    h.load_state_dict(ckpt["state_dict"])
    h.eval()
    heads.append(h)
print(f"Heads loaded. NUM_CLASSES={NUM_CLASSES}, EMBED_DIM={EMBED_DIM}")


## 5. Audio + windowing helpers


In [ ]:
SR          = 32000
CLIP_SEC    = 5
N_SAMPLES   = SR * CLIP_SEC
WINDOW_SEC  = CLIP_SEC
N_WINDOWS   = 60 // WINDOW_SEC   # 12
GAUSSIAN_KERNEL = np.array([0.1, 0.2, 0.4, 0.2, 0.1])


def load_audio(path):
    wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    return wav.astype(np.float32)


def file_to_chunks(path):
    """Load file, pad/truncate to 60s, reshape to (12, N_SAMPLES)."""
    wav = load_audio(path)
    target = N_WINDOWS * N_SAMPLES
    if len(wav) < target:
        wav = np.pad(wav, (0, target - len(wav)))
    else:
        wav = wav[:target]
    return wav.reshape(N_WINDOWS, N_SAMPLES).astype(np.float32)


def smooth_windows(logits):
    return convolve1d(logits, GAUSSIAN_KERNEL, axis=0, mode="nearest")


## 6. Inference function

Run Perch on 12 windows at once (batched), mean-pool freq dim, run all 10 heads,
average their logits, smooth across time, sigmoid.


In [ ]:
@torch.no_grad()
def predict_file(path):
    chunks = file_to_chunks(path)                                  # (12, 160000)
    # Perch — get spatial_embedding (12, 16, 4, 1536)
    spatial = sess.run(None, {INPUT_NAME: chunks})[SPATIAL_IDX]
    spatial_pooled = spatial.mean(axis=2)                          # mean over freq → (12, 16, 1536)
    spatial_t = torch.from_numpy(spatial_pooled).to(DEVICE)
    # Ensemble of 10 heads — average logits in logit space
    logits = (sum(h(spatial_t) for h in heads) / len(heads)).cpu().numpy()  # (12, 234)
    logits = smooth_windows(logits)                                # Gaussian smooth across 12 windows
    return 1.0 / (1.0 + np.exp(-logits))                           # sigmoid → probs


# Smoke test on one file
TEST_DIR  = DATA_DIR / "test_soundscapes"
TRAIN_DIR = DATA_DIR / "train_soundscapes"
test_files = sorted(TEST_DIR.glob("*.ogg")) if TEST_DIR.is_dir() else []
if not test_files:
    test_files = sorted(TRAIN_DIR.glob("*.ogg"))[:5]
    print(f"[fallback] using {len(test_files)} files from train_soundscapes/")
else:
    print(f"Found {len(test_files)} test files")

probs0 = predict_file(test_files[0])
print(f"sample output shape: {probs0.shape}   range: [{probs0.min():.3f}, {probs0.max():.3f}]")


## 7. Run inference on all test files


In [ ]:
all_rows, all_probs = [], []
t0 = time.time()
for f in tqdm(test_files, desc="infer"):
    basename = f.stem
    probs    = predict_file(f)
    end_secs = np.arange(1, N_WINDOWS + 1) * WINDOW_SEC      # 5, 10, ..., 60
    for k in range(N_WINDOWS):
        all_rows.append(f"{basename}_{end_secs[k]}")
        all_probs.append(probs[k])
all_probs = np.stack(all_probs)
print(f"{len(all_rows)} rows in {time.time()-t0:.1f}s")


## 8. Build submission.csv with correct column order


In [ ]:
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
all_species_in_order = [c for c in sample_sub.columns if c != "row_id"]

pred_df = pd.DataFrame(all_probs, columns=species)
pred_df.insert(0, "row_id", all_rows)

# Reindex to the full taxonomy; species we didn't train on get a uniform prior
sub = pred_df.set_index("row_id").reindex(columns=all_species_in_order)
sub = sub.fillna(1.0 / len(all_species_in_order)).clip(0.0, 1.0).reset_index()

assert list(sub.columns) == list(sample_sub.columns), "Column order mismatch!"
assert sub["row_id"].is_unique
assert not sub.isna().any().any()

sub.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH}  shape={sub.shape}  size={OUT_PATH.stat().st_size/1024:.1f} KB")
sub.head(3)


## What's next

Locally this is just a pipeline-check (we run on 5 train soundscape files since
`test_soundscapes/` is empty). For the real LB score:

1. Upload all 10 checkpoints to your Kaggle dataset (we use `harishteens/birdclef-2026-baseline-ckpt`).
2. Use the Kaggle kernel at `kaggle-uploads/baseline-submit/` (logic mirrors this notebook with Kaggle paths).
3. Submit via `kaggle competitions submit -c birdclef-2026 -k <kernel> -v <version> -f submission.csv -m "..."`.

Current LB: **0.842** (10-model combined ensemble).
